# FINS notebook 9 — audit of PG8's pyqg emulators; LieNLSD on exp60 (Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp65 EA1–EA3, exp66 LS1–LS2, 10.10) before any of these runs.
* cell 2 (exp65): for each of PG8's six FNO2d emulators (f and beta, seeds 42–44; read from
  `MyDrive/fins_runs/results/pyqg/fno2d_*.pt`, trained by notebook 8), FINS through the emulator on its own rollouts,
  then its one-step error on transformed pyqg states. About 2–4 min per emulator.
* cell 3 (exp66): LieNLSD as published on exp60's data, a in {1.25, 1.5, 1.75}, seeds 42–51 (30 runs, a few minutes
  each).
* cell 4 shows progress; cell 5 zips the json and logs to `fins_nb9.zip` (small: upload this one to the chat), in
  /content and on My Drive.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down). After a
disconnect rerun cell 1 and the cell that stopped: finished runs are skipped.

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp65, exp66), pyqg from GitHub, PG8's emulators on Drive
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(R, exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp66(z):
    with zipfile.ZipFile(z) as zz:
        names = zz.namelist()
        return all(f'fins_repo/experiments/{f}' in names for f in ('exp65_emulator_audit.py', 'exp66_lienlsd_aniso.py'))


ZIP_PATH = newest('fins_repo_full*.zip', has_exp66)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp65 and exp66: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/pyqg/pyqg.git', 'sympy'],
               check=False)
for mod in ('exp65_emulator_audit', 'exp66_lienlsd_aniso'):
    chk = subprocess.run([sys.executable, '-c', f'import {mod}'], cwd=EXPDIR, capture_output=True, text=True)
    if chk.returncode:
        raise SystemExit(f'{mod} does not import here:\n' + chk.stderr[-2000:])
print('exp65 and exp66 import')
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')
REGIMES, SEEDS = ('f', 'beta'), (42, 43, 44)
missing = [f'{r} s{s}' for r in REGIMES for s in SEEDS
           if not os.path.exists(os.path.join(R, 'pyqg', f'fno2d_{r}_q_n128_s{s}.pt'))]
print('PG8 emulators: all six present' if not missing else 'MISSING emulators (notebook 8 saved them under '
      'MyDrive/fins_runs/results/pyqg): ' + ', '.join(missing))
AUDIT, LNS = os.path.join(R, 'audit_runs'), os.path.join(R, 'lienlsd_aniso_runs')
for d in (AUDIT, LNS):
    os.makedirs(os.path.join(d, 'logs'), exist_ok=True)


def go(runs, stem, cmd, keys):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(runs, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if rc == 0 else lines[-6:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:150] for k in keep[-6:]], sep='\n   ')

In [ ]:
# 2. exp65: each PG8 emulator -- FINS on its own rollouts, then the transformed-input errors
for reg in REGIMES:
    for s in SEEDS:
        for what in ('fins', 'consequence'):
            stem = os.path.join(AUDIT, f'audit_{what}_{reg}_s{s}')
            go(AUDIT, stem, ['exp65_emulator_audit.py', what, '--regime', reg, '--seed', str(s), '--save', stem],
               ('quotients', 'rotation plane', 'ratio', 'own rollouts'))
left = [f'{w} {r} s{s}' for w in ('fins', 'consequence') for r in REGIMES for s in SEEDS
        if not os.path.exists(os.path.join(AUDIT, f'audit_{w}_{r}_s{s}.json'))]
print('all 12 audit results present' if not left else 'MISSING (see the logs above): ' + ', '.join(left))

In [ ]:
# 3. exp66: LieNLSD on exp60, a in {1.25, 1.5, 1.75}, seeds 42-51
for a in ('1.25', '1.5', '1.75'):
    for s in range(42, 52):
        stem = os.path.join(LNS, f'lienlsd_a{a}_s{s}')
        go(LNS, stem, ['exp66_lienlsd_aniso.py', 'one', '--a', a, '--seed', str(s), '--save', stem], ('fitted',))
left = [f'a {a} s{s}' for a in ('1.25', '1.5', '1.75') for s in range(42, 52)
        if not os.path.exists(os.path.join(LNS, f'lienlsd_a{a}_s{s}.json'))]
print('all 30 LieNLSD results present' if not left else 'MISSING (see the logs above): ' + ', '.join(left))

In [ ]:
# 4. progress
n1 = len(glob.glob(os.path.join(AUDIT, 'audit_*.json')))
n2 = len(glob.glob(os.path.join(LNS, 'lienlsd_*.json')))
print(f'audit: {n1}/12 results; LieNLSD: {n2}/30 results')

In [ ]:
# 5. copy the json and logs file by file to local disk (remount if Drive drops); fins_nb9.zip in /content and on Drive
local = '/content/fins_nb9'
shutil.rmtree(local, ignore_errors=True)
files = []
for d in (AUDIT, LNS):
    files += glob.glob(os.path.join(d, '*.json')) + glob.glob(os.path.join(d, 'logs', '*.log'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb9.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb9.zip')
    print('copied to MyDrive: fins_nb9.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb9.zip from the file pane:', e)